# Solution — Multi-API Kafka Pipeline: Consumers

Two consumers in this notebook:
- **Weather Consumer** — reads from `weather` topic, prints formatted conditions per city
- **Bikes Consumer** — reads from `bikes` topic, tracks bike changes per station and fires operations alerts

> Run both consumers **before** starting the producers.

---
## Consumer 1 — Weather

In [ ]:
from kafka import KafkaConsumer
import json

BROKERS       = ['course-kafka:9092']
WEATHER_TOPIC = 'weather'

In [ ]:
# group_id='weather-monitor' — offset is committed automatically, so on restart
# the consumer picks up exactly where it left off instead of re-reading from the beginning.
weather_consumer = KafkaConsumer(
    WEATHER_TOPIC,
    group_id                = 'weather-monitor',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

In [ ]:
for message in weather_consumer:

    # Deserialize: bytes → str → dict
    d = json.loads(message.value.decode('utf-8'))

    # f-string with fixed-width fields keeps the output aligned even when
    # city names have different lengths.
    print(
        f"📍 {d['city']:<12} | "
        f"🌡  {d['temperature']:>5.1f}°C | "
        f"💨 {d['windspeed']:>5.1f} km/h | "
        f"{d['condition']}"
    )

---
## Consumer 2 — Bikes (with operations alerts)

In [ ]:
from kafka import KafkaConsumer
import json

BROKERS     = ['course-kafka:9092']
BIKES_TOPIC = 'bikes'

# Alert thresholds for the operations team
LOW_BIKES_LIMIT = 2    # <= 2 bikes left  -> station is almost empty, bring bikes
LOW_DOCKS_LIMIT = 2    # <= 2 free docks  -> station is almost full, take bikes away

In [ ]:
bikes_consumer = KafkaConsumer(
    BIKES_TOPIC,
    group_id                = 'bikes-monitor',
    bootstrap_servers       = BROKERS,
    auto_commit_interval_ms = 1000
)

In [ ]:
# prev_bikes = running STATE: the last bikes_available we saw for each station.
# Empty at start — the first message per station has nothing to compare against.
prev_bikes = {}

for message in bikes_consumer:

    # Deserialize: bytes → str → dict
    d     = json.loads(message.value.decode('utf-8'))
    sid   = d['station_id']
    name  = d['name']
    bikes = d['bikes_available']
    docks = d['docks_available']
    fill  = bikes / d['capacity'] * 100 if d['capacity'] else 0   # guard against capacity 0

    if sid not in prev_bikes:
        change_str = 'new'                              # first message for this station
    else:
        change_str = f"{bikes - prev_bikes[sid]:+d}"    # +d always shows the sign: +2 / -3 / +0

    print(f"🚲 {name:<30.30} | {bikes:>2}/{d['capacity']:<2} bikes ({fill:>3.0f}%) | change: {change_str}")

    # Operations alerts
    if bikes <= LOW_BIKES_LIMIT:
        print(f"     ⚠️  LOW BIKES: {name} has only {bikes} bikes — send a truck with bikes!")
    if docks <= LOW_DOCKS_LIMIT:
        print(f"     ⚠️  STATION FULL: {name} has only {docks} free docks — take bikes away!")

    # Update the state AFTER printing, so the next message compares against this one.
    prev_bikes[sid] = bikes